# 🧪 DervGen GPU Quickstart & Micro-Benchmark

This notebook verifies GPU acceleration, FP32 / FP16 tensor performance (TFLOPS), and VRAM bandwidth in your Colab environment.

In [ ]:
import time
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    props = torch.cuda.get_device_properties(device)
    print(f"Device: {props.name}")
    print(f"Total Memory: {props.total_memory / (1024**3):.2f} GB")
    print(f"Multi-Processors: {props.multi_processor_count}")
    print(f"Compute Capability: {props.major}.{props.minor}")
else:
    device = torch.device("cpu")
    print("Running on CPU.")

## Matrix Multiplication (GEMM) Benchmark (FP32 & FP16)

In [ ]:
def benchmark_matmul(size=8192, dtype=torch.float32, iterations=10):
    a = torch.randn(size, size, device=device, dtype=dtype)
    b = torch.randn(size, size, device=device, dtype=dtype)
    
    # Warmup
    for _ in range(3):
        _ = torch.matmul(a, b)
    if device.type == "cuda":
        torch.cuda.synchronize()
        
    # Benchmark
    start = time.perf_counter()
    for _ in range(iterations):
        _ = torch.matmul(a, b)
    if device.type == "cuda":
        torch.cuda.synchronize()
    duration = (time.perf_counter() - start) / iterations
    
    # 2 * N^3 FLOPs
    tflops = (2.0 * (size ** 3)) / (duration * 1e12)
    print(f"Matrix Size: {size}x{size} | Dtype: {dtype} | Latency: {duration*1000:.2f} ms | Throughput: {tflops:.2f} TFLOPS")

if device.type == "cuda":
    print("--- FP32 GEMM Benchmark ---")
    benchmark_matmul(size=4096, dtype=torch.float32)
    
    print("--- FP16 GEMM Benchmark ---")
    benchmark_matmul(size=4096, dtype=torch.float16)
else:
    print("Skipping GPU GEMM benchmark (CUDA not active).")